# PyTorch MLP Training

This notebook implements a PyTorch-based multilayer perceptron for the binary diabetes risk prediction task.

The goal is to compare a neural network model against the tuned scikit-learn Random Forest baseline using the same dataset, target definition, train/test split, preprocessing strategy, and evaluation metrics.

This notebook focuses on the core deep learning training pipeline. Calibration and uncertainty-aware analysis will be explored in a later notebook.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

In [3]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

## Load Dataset

The dataset is loaded from the local `data/` directory.

As in the previous notebooks, the original three-class target is converted into a binary clinical risk prediction target:

- 0: No diabetes
- 1: At risk, including prediabetes or diabetes

This keeps the PyTorch experiment directly comparable to the scikit-learn baseline.

In [4]:
DATA_PATH = Path("../data/diabetes_012_health_indicators_BRFSS2015.csv")

def load_diabetes_data(data_path=DATA_PATH):
    if not data_path.is_file():
        raise FileNotFoundError(
            f"Dataset not found at {data_path}. "
            "Please place the BRFSS 2015 diabetes CSV file in the data/ directory."
        )
    return pd.read_csv(data_path)

df = load_diabetes_data()

print(df.shape)
print(df.head())


(253680, 22)
   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3           0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4           0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      5.0      18.0      15.0       1.0  0.0   9.0        

In [5]:
# Convert the original 3-class target into a binary clinical risk label
df["Diabetes_binary"] = (df["Diabetes_012"] > 0).astype(int)

X = df.drop(columns=["Diabetes_binary", "Diabetes_012"])
y = df["Diabetes_binary"]

X.shape, y.shape

((253680, 21), (253680,))